In [ ]:
import glob, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Load all daily draw CSVs ──────────────────────────────────────────────────
files = glob.glob('daily_draws/*.csv')
daily_raw = pd.concat([
    pd.read_csv(f).assign(draw_date=os.path.splitext(os.path.basename(f))[0])
    for f in files
])
daily_raw.rename(columns={'time [UTC]': 'date', 'value': 'daily'}, inplace=True)
daily_raw['date']      = pd.to_datetime(daily_raw['date'],      format='mixed').dt.date
daily_raw['draw_date'] = pd.to_datetime(
    daily_raw['draw_date'].str.replace('draw_', ''), format='mixed'
).dt.date
daily_pivot     = daily_raw.pivot_table(index='date', columns='draw_date',
                                        values='daily', aggfunc='first')
daily_draw_cols = daily_pivot.columns.tolist()

# ── Weekly benchmarks (rescaled final series) ────────────────────────────────
weekly_final = pivot[draw_cols].mean(axis=1)
weekly_final = weekly_final / weekly_final.max() * 100

# ── Drop partial-week benchmarks ─────────────────────────────────────────────
# Google Trends dates are week-START (Sunday). The window is [wk, wk+6].
# Benchmarks where the full 7-day window falls outside the daily sample produce
# a degenerate constraint (mean of <7 days) and must be dropped.
daily_start = pd.to_datetime(daily_pivot.index.min())
daily_end   = pd.to_datetime(daily_pivot.index.max())
weekly_final = weekly_final[
    (pd.to_datetime(weekly_final.index) >= daily_start) &
    (pd.to_datetime(weekly_final.index) + pd.Timedelta(days=6) <= daily_end)
]
print(f"Benchmarks after trimming partial weeks: {len(weekly_final)}")
print(f"  First: {pd.to_datetime(weekly_final.index[0]).date()}  "
      f"Last:  {pd.to_datetime(weekly_final.index[-1]).date()}")

# ── Scale reconciliation ─────────────────────────────────────────────────────
# Dates are week-START → window is [wk, wk+6]  (FIX: was wk-6..wk)
def estimate_scale_constant(indicator: pd.Series, benchmarks: pd.Series) -> float:
    dates = pd.to_datetime(indicator.index)
    weekly_means = []
    for wk in pd.to_datetime(benchmarks.index):
        mask = (dates >= wk) & (dates <= wk + pd.Timedelta(days=6))  # week-START
        vals = indicator[mask]
        weekly_means.append(vals.mean() if len(vals) > 0 else np.nan)
    weekly_means = pd.Series(weekly_means, index=benchmarks.index)
    valid = weekly_means.notna()
    return float(benchmarks[valid].mean() / weekly_means[valid].mean())


# ── TRUE Denton proportional adjustment ──────────────────────────────────────
# Solves globally:
#   min  Σ_{t=2}^{T} ( X_t/x_t - X_{t-1}/x_{t-1} )²
#   s.t. (1/n_k) Σ_{t ∈ w_k} X_t = B_k   for every benchmark week k
#
# Dates are week-START → window is [wk, wk+6]  (FIX: was wk-6..wk)
# KKT solution:  p* = p₀ + Q⁺ A' (A Q⁺ A')⁻¹ (b_sum − A p₀),  p₀ = 1
def denton_proportional(indicator: pd.Series, benchmarks: pd.Series) -> pd.Series:
    x     = indicator.values.astype(float)
    n     = len(x)
    dates = pd.to_datetime(indicator.index)
    m     = len(benchmarks)

    J       = np.zeros((m, n))
    n_k_arr = np.zeros(m)
    for k, wk in enumerate(pd.to_datetime(benchmarks.index)):
        mask        = (dates >= wk) & (dates <= wk + pd.Timedelta(days=6))  # week-START
        n_k_arr[k]  = mask.sum()
        J[k, mask]  = 1.0

    A     = J * x[np.newaxis, :]
    b_sum = benchmarks.values.astype(float) * n_k_arr

    D  = np.diff(np.eye(n), axis=0)
    Q  = D.T @ D

    p0    = np.ones(n)
    Qp    = np.linalg.pinv(Q)
    M     = A @ Qp @ A.T
    lam   = np.linalg.lstsq(M, b_sum - A @ p0, rcond=None)[0]
    p_opt = p0 + Qp @ A.T @ lam

    return pd.Series(p_opt * x, index=indicator.index)


# ── Method 1: Naive average + Denton ─────────────────────────────────────────
naive_avg    = daily_pivot[daily_draw_cols].mean(axis=1)
C_naive      = estimate_scale_constant(naive_avg, weekly_final)
print(f"\nScale constant (naive avg):    C = {C_naive:.4f}")
naive_denton = denton_proportional(naive_avg * C_naive, weekly_final)
naive_denton = naive_denton / naive_denton.max() * 100


# ── Method 2: Rescale to consensus peak, then average + Denton ───────────────
daily_peaks          = daily_pivot[daily_draw_cols].idxmax()
consensus_peak_daily = daily_peaks.mode()[0]
n_agree = (daily_peaks == consensus_peak_daily).sum()
print(f"Daily consensus peak: {consensus_peak_daily}  ({n_agree}/{len(daily_draw_cols)} draws)")

daily_rescaled = daily_pivot[daily_draw_cols].copy().astype(float)
for col in daily_draw_cols:
    if daily_peaks[col] != consensus_peak_daily:
        val = daily_rescaled.loc[consensus_peak_daily, col]
        if val > 0:
            daily_rescaled[col] = daily_rescaled[col] * (100.0 / val)
            print(f"  Rescaled {col} (peaked at {daily_peaks[col]}, had {val:.1f} at consensus)")

rescaled_avg    = daily_rescaled.mean(axis=1)
C_rescaled      = estimate_scale_constant(rescaled_avg, weekly_final)
print(f"Scale constant (rescaled avg): C = {C_rescaled:.4f}")
rescaled_denton = denton_proportional(rescaled_avg * C_rescaled, weekly_final)
rescaled_denton = rescaled_denton / rescaled_denton.max() * 100


# ── Sanity check: constraint should hold to ~1e-10 ───────────────────────────
# Pick a benchmark week in the middle and verify Denton mean ≈ benchmark.
mid_idx  = len(weekly_final) // 2
wk_check = pd.to_datetime(weekly_final.index[mid_idx])
mask_chk = (pd.to_datetime(naive_denton.index) >= wk_check) & \
           (pd.to_datetime(naive_denton.index) <= wk_check + pd.Timedelta(days=6))

# naive_denton was rescaled by /max*100, so rescale benchmark the same way
scale    = naive_denton.max() / 100   # ≈ 1 since we divided by max
denton_mean_raw = (naive_denton[mask_chk] / 100 * naive_denton.max()).mean()  # raw scale
bench_raw       = weekly_final.iloc[mid_idx] * C_naive * naive_avg.max() / 100

print(f"\n── Sanity check (week starting {wk_check.date()}) ──")
wk_check2 = pd.Timestamp('2024-06-02')  # fixed mid-sample week from the suggestion
mask2 = (pd.to_datetime(naive_denton.index) >= wk_check2) & \
        (pd.to_datetime(naive_denton.index) <= wk_check2 + pd.Timedelta(days=6))
if mask2.sum() > 0:
    print(f"  naive_denton mean in week:  {naive_denton[mask2].mean():.6f}")
    print(f"  weekly_final benchmark:     {weekly_final.loc['2024-06-02']:.6f}")
    print(f"  n days in window:           {mask2.sum()}")
    print(f"  diff (should be ~0 after same rescaling): "
          f"{abs(naive_denton[mask2].mean() - weekly_final.loc['2024-06-02']):.2e}")


# ── Plot comparison ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(naive_denton.index,    naive_denton.values,
        color='red',      lw=1.5, ls='--', label='Naive avg + Denton')
ax.plot(rescaled_denton.index, rescaled_denton.values,
        color='darkblue', lw=2,            label='Rescaled avg + Denton')
ax.set(xlabel='Date', ylabel='Index (max = 100)',
       title=f'Daily Draws: Both Denton-adjusted ({len(daily_draw_cols)} draws)')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

diff = (rescaled_denton - naive_denton).abs()
print(f"\nMax absolute difference:  {diff.max():.2f}")
print(f"Mean absolute difference: {diff.mean():.2f}")

In [ ]:
# ── Diagnose: are benchmark dates week-END or week-START? ────────────────────
bench_dates = pd.to_datetime(weekly_final.index)
daily_dates = pd.to_datetime(daily_pivot.index)

print("=== Sample benchmark dates ===")
for d in bench_dates[:5]:
    print(f"  {d.date()}  →  {d.day_name()}")

print(f"\nWeekday distribution across all {len(bench_dates)} benchmark dates:")
print(bench_dates.day_name().value_counts().sort_index())

print(f"\nDaily data range:     {daily_dates.min().date()} → {daily_dates.max().date()}")
print(f"Benchmark date range: {bench_dates.min().date()} → {bench_dates.max().date()}")

# For each benchmark date test both interpretations
print("\n=== Coverage check (first 5 benchmarks) ===")
print(f"{'Date':<14} {'Weekday':<12} "
      f"{'END window (d-6..d)':<23} {'START window (d..d+6)'}")
for d in bench_dates[:5]:
    n_end   = ((daily_dates >= d - pd.Timedelta(days=6)) & (daily_dates <= d)).sum()
    n_start = ((daily_dates >= d) & (daily_dates <= d + pd.Timedelta(days=6))).sum()
    print(f"  {str(d.date()):<12}  {d.day_name():<12}  {n_end:>3} daily obs"
          f"                 {n_start:>3} daily obs")

# Aggregate over all benchmarks
end_total   = sum(((daily_dates >= d - pd.Timedelta(days=6)) & (daily_dates <= d)).sum()
                  for d in bench_dates)
start_total = sum(((daily_dates >= d) & (daily_dates <= d + pd.Timedelta(days=6))).sum()
                  for d in bench_dates)

print(f"\nTotal daily obs matched:")
print(f"  Treating dates as week-END   (d-6 .. d  ): {end_total}")
print(f"  Treating dates as week-START (d   .. d+6): {start_total}")
print(f"\n→ Higher count = correct convention.")
print(f"  (Google Trends weekly dates are typically the SUNDAY ending the week)")

# Data Science Tools and Ecosystem


In this notebook, Data Science Tools and Ecosystem are summarized.

**Objectives:**

- List popular languages for Data Science.
- Introduce commonly used libraries in Data Science.
- Explore examples of evaluating arithmetic expressions in Python.
- Understand how to convert minutes to hours using Python.
- Summarize the Data Science tools and ecosystem.


Some of the popular languages that Data Scientists use are:

1. Python
2. R
3. SQL
4. Julia
5. Scala


Some of the commonly used libraries by Data Scientists include:

1. NumPy
2. Pandas
3. Matplotlib
4. Scikit-Learn
5. TensorFlow


| Data Science Tools    |
|-----------------------|
| Jupyter Notebook      |
| RStudio               |
| Visual Studio Code    |


### Examples of Evaluating Arithmetic Expressions in Python

In Python, you can perform various arithmetic operations. Here are some examples:

1. **Addition:**
   ```python
   result = 5 + 3
   # result will be 8
result = 10 - 4
# result will be 6


In [6]:
# This a simple arithmetic expression to mutiply then add integers
(3*4)+5

17

In [8]:
#This will convert 200 minutes to hours by diving by 60
200/60

3.3333333333333335

## Author
Andrea Lamacchia


## Robustness Check: Denton on Naive Average vs Denton on Rescaled Average

**Working assumption:** weekly Gtrends = within-week **mean** of the underlying daily query share × an unknown multiplicative constant.  
The constant is estimated by aligning the two series on the full-sample mean (scale reconciliation) before applying the Denton proportional adjustment.  
Pro-rata via weekly **sum** (as often coded by default) is wrong here and throws away cross-week information.

Two methods compared:
1. **Naive avg + Denton** — average the raw daily draws, reconcile scale, apply Denton.
2. **Rescaled avg + Denton** — rescale each draw to a consensus peak first, then average, reconcile scale, apply Denton.